In this notebook, I want mainly want to try to feature engineer up to degree two and see if interactions between stats have any more predictive power. The thing is, this level of dimensionality (around 3000 features) will cause extreme overfitting of the data in a non-regularized logistic regressor, but forward feature selection to select features is difficult due to extreme computational requirements when the degree gets that high. So, I will try to use lasso regression to do implicit feature selection on this higher dimension dataset to try to reduce some dimensionality more feasibly. I will not go beyond degree two since degree three already jumps from ~3000 features to ~79000 features which is computationally infeasible.

In [1]:
import pandas as pd
df = pd.read_csv("../../data/aggregated_rolling_features.csv")
df.shape

(5539, 97)

In [2]:
to_drop = []
to_drop.extend([f"team{i}_player{k}_id" for i in range(1,3) for k in range(1,6)])
to_drop.extend([f"team{i}_id" for i in range(1,3)])
to_drop.extend([f"team{i}" for i in range(1,3)])
to_drop.extend(["tournament", "match_id", "game_id", "map_id", "map_name", "datetime", "Unnamed: 0"])
df = df.drop(to_drop, axis=1)
df

,team1_win,bestOf,team1_previous_10_average_map_score,team2_previous_10_average_map_score,previous_10_games_team1_average_kills,previous_10_games_team1_std_kills,previous_10_games_team1_range_kills,previous_10_games_team1_max_kills,previous_10_games_team1_min_kills,previous_10_games_team1_median_kills,...,previous_10_games_team2_range_kast,previous_10_games_team2_max_kast,previous_10_games_team2_min_kast,previous_10_games_team2_median_kast,previous_10_games_team2_average_kddiff,previous_10_games_team2_std_kddiff,previous_10_games_team2_range_kddiff,previous_10_games_team2_max_kddiff,previous_10_games_team2_min_kddiff,previous_10_games_team2_median_kddiff
0,0,3.0,13.000000,13.000000,16.000000,0.000000e+00,0.0,16.000000,16.000000,16.000000,...,0.00,70.000000,70.000000,70.000000,1.000000,0.000000,0.0,1.000000,1.000000,1.000000
1,1,3.0,12.333333,12.333333,15.600000,4.127953e+00,10.0,22.000000,12.000000,13.000000,...,25.00,83.300000,58.300000,58.300000,-0.600000,3.136877,9.0,4.000000,-5.000000,0.000000
2,1,3.0,11.600000,11.600000,15.100000,2.083267e+00,5.5,18.500000,13.000000,14.000000,...,17.30,75.000000,57.700000,60.100000,-1.600000,5.885576,17.5,8.500000,-9.000000,-1.500000
3,1,3.0,11.571429,11.571429,14.612903,1.776357e-15,0.0,14.612903,14.612903,14.612903,...,0.00,71.535484,71.535484,71.535484,-0.032258,0.000000,0.0,-0.032258,-0.032258,-0.032258
4,1,3.0,10.666667,10.666667,13.600000,4.363485e+00,13.0,21.000000,8.000000,13.000000,...,33.30,60.000000,26.700000,33.300000,-9.000000,2.280351,6.0,-6.000000,-12.000000,-9.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5534,0,3.0,13.400000,9.500000,14.140000,9.200000e-01,2.6,15.400000,12.800000,14.200000,...,6.89,77.400000,70.510000,76.760000,0.780000,2.465279,6.4,3.500000,-2.900000,1.800000
5535,0,3.0,9.700000,11.100000,14.680000,1.175415e+00,3.3,16.600000,13.300000,14.900000,...,6.96,78.300000,71.340000,76.820000,0.920000,2.741095,7.0,3.700000,-3.300000,2.300000
5536,1,3.0,11.000000,10.700000,14.440000,6.590903e-01,1.7,15.400000,13.700000,14.600000,...,6.96,79.800000,72.840000,75.820000,1.140000,2.793278,7.1,3.900000,-3.200000,2.700000
5537,0,5.0,11.400000,12.700000,14.780000,2.594147e+00,6.8,17.500000,10.700000,14.900000,...,5.71,83.730000,78.020000,82.000000,4.280000,3.398470,8.7,7.800000,-0.900000,5.200000


In [3]:
X = df.drop("team1_win", axis=1)
y = df.team1_win

In [ ]:
X_numerical = X.drop("bestOf", axis=1)
X_categorical = X.bestOf
X_categorical = pd.get_dummies(drop_first=True,dtype=int,prefix="bestOf")

Let's get the feature engineered set before standardizing

In [7]:
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2,include_bias=False)
X_numerical_poly_prime = poly.fit_transform(X_numerical)
X_numerical_poly = pd.DataFrame(X_numerical_poly_prime, columns=poly.get_feature_names_out(X_numerical.columns))
X_poly = pd.concat([X_numerical_poly, X_categorical], axis=1)
X_poly.shape

(5539, 2850)

Standardize

In [9]:
from sklearn.preprocessing import StandardScaler
from math import ceil
stnd = StandardScaler().set_output(transform="pandas")
#Split
split_point = ceil(len(df) * 0.8)
train_numerical = X_numerical.iloc[:split_point] # 0 to split_point - 1
test_numerical = X_numerical.iloc[split_point:] # split_point to len(df)
train_poly = X_poly.iloc[:split_point] # 0 to split_point - 1
test_poly = X_poly.iloc[split_point:] # split_point to len(df)
train_cat = X_categorical.iloc[:split_point]
test_cat = X_categorical.iloc[split_point:]
y_train = y.iloc[:split_point]
y_test = y.iloc[split_point:]
#Standardize
train_numerical = stnd.fit_transform(train_numerical)
test_numerical = stnd.transform(test_numerical)
train_poly = stnd.fit_transform(train_poly)
test_poly = stnd.transform(test_poly)
#Concat
X_train = pd.concat([train_numerical, train_cat], axis=1)
X_test = pd.concat([test_numerical, test_cat], axis=1)
X_poly_train = pd.concat([train_poly, train_cat], axis=1)
X_poly_test = pd.concat([test_poly, test_cat], axis=1)
#Show results
print("X_train shape", X_train.shape)
print("X_test shape", X_test.shape)
print("X_poly_train shape", X_poly_train.shape)
print("X_poly_test shape", X_poly_test.shape)

X_train shape (4432, 75)
X_test shape (1107, 75)
X_poly_train shape (4432, 2851)
X_poly_test shape (1107, 2851)


# First, let's get a baseline view of how lasso performs before we feature engineer